# VisClick — Part E (step 6): desktop fine-tune from auto-labels

**Prerequisite this session:**
- `<DRIVE>/weights/baseline_source/best_source_v8s.pt` exists (produced by `05_train_source.ipynb`).
- `<DRIVE>/data/desktop/raw/` is **either** populated with your own desktop screenshots **or** left empty — in the latter case, cell 6.1 auto-seeds it from `samples/desktop_seed/` in the cloned repo (50 personal screenshots of VS Code / Excel / Chrome shipped with the project).

**This notebook** (`VisClick_Detailed_Plan.md` §E):
1. Mount Drive → `git pull` → install Ultralytics.
2. **Auto-label** the raw screenshots with `best_source_v8s.pt` (the source-domain "teacher"). Writes one YOLO-format `.txt` per image to `<DRIVE>/data/desktop/labels/`. Idempotent (skips already-labelled images).
3. **Sanity preview**: save a 9-image grid with predicted boxes overlaid to `<DRIVE>/reports/figures/desktop_autolabel_preview.png`. Open it and eyeball — bad labels can be fixed in Roboflow / LabelImg later.
4. **Assemble** `/content/desktop_finetune/` (local) with a deterministic 70/15/15 split (≈ 35/8/7 of 50). Persist labels **and** images as small `.tar.gz` bundles to `<DRIVE>/data/desktop_finetune_bundles/` so any new Colab tab restores in seconds.
5. **Head-only fine-tune** YOLOv8s starting from `best_source_v8s.pt`: `freeze=10` (lock backbone), `epochs=20`, `batch=8`, `imgsz=640`, `lr0=1e-3`. Saves to `<DRIVE>/weights/desktop_finetune/run1/`. Resume-aware (rerun the cell after a disconnect).
6. **Save** stable name `desktop_finetune/best_desktop_v8s.pt` for `07_demo.ipynb`.
7. **Eval** on the held-out 7-image test split + sanity predictions + per-class CSV at `<DRIVE>/reports/tables/desktop_finetune_results.csv` & `desktop_finetune_per_class.csv`.

**Caveat — pseudo-labels.** The teacher's mAP@.5 on its own test set is 0.45, so the auto-labels are noisy (~50% wrong). Fine-tuning on noisy labels is "self-training" / "knowledge distillation": it adapts the head to desktop UI density and themes (dark VS Code, Excel ribbon, Chrome) but does **not** introduce class knowledge the teacher lacks. Replacing the auto-labels with a pass through Roboflow afterwards is the recommended next step for the dissertation's headline number.

**Report:** every step prints `REPORT ...` lines for `VisClick_Report_Data_Form.md` §4 row *Desktop fine-tune* and §4.5 per-class.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "https://github.com/HiranMadhu/visclick.git"
ROOT = "/content/visclick"
if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", REPO, ROOT], check=True)
    print("Cloned to", ROOT)
else:
    subprocess.run(["git", "-C", ROOT, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", ROOT, "pull", "--rebase", "origin", "main"], check=False)
    print("Pulled latest in", ROOT)
print("REPORT git_head =", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics", "opencv-python"], check=False)
import torch, ultralytics
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("ultralytics:", ultralytics.__version__)
print("REPORT env | torch =", torch.__version__, "| cuda =", torch.cuda.is_available(), "| ultralytics =", ultralytics.__version__)

## 6.1 — Auto-label raw screenshots with `best_source_v8s.pt`

Reads every image in `<DRIVE>/data/desktop/raw/` (recursively, PNG/JPG/JPEG) and runs the source-baseline teacher at `conf=0.20` (deliberately permissive — the head will re-learn during fine-tune).

For each image `<stem>.<ext>` we write a YOLO-format label `<DRIVE>/data/desktop/labels/<stem>.txt` (one line per detection: `class_id cx cy w h`, all normalized 0–1). **Idempotent** — images that already have a label file are skipped, so you can drop more screenshots into `raw/` later and re-run the cell.


In [ ]:
import os, glob, shutil, time
from collections import Counter
from ultralytics import YOLO

DRIVE   = "/content/drive/MyDrive/visclick"
RAW     = os.path.join(DRIVE, "data", "desktop", "raw")
LABELS  = os.path.join(DRIVE, "data", "desktop", "labels")
TEACHER = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
SEED    = "/content/visclick/samples/desktop_seed"
os.makedirs(RAW, exist_ok=True)
os.makedirs(LABELS, exist_ok=True)

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]
EXTS = (".png", ".jpg", ".jpeg")
CONF = 0.20
IOU  = 0.50

assert os.path.isfile(TEACHER), f"Missing teacher weights: {TEACHER}. Run 05 first."

def _is_img(p):
    return p.lower().endswith(EXTS)

raw_imgs = sorted(p for p in glob.glob(os.path.join(RAW, "*")) if _is_img(p))
if len(raw_imgs) == 0 and os.path.isdir(SEED):
    seed_imgs = sorted(p for p in glob.glob(os.path.join(SEED, "*")) if _is_img(p))
    print(f"REPORT step = SEED | dst = {RAW} (empty) | src = {SEED} | n = {len(seed_imgs)}")
    t0 = time.time()
    for sp in seed_imgs:
        dp = os.path.join(RAW, os.path.basename(sp))
        if not os.path.isfile(dp):
            shutil.copy2(sp, dp)
    print(f"REPORT step = SEED | copied | elapsed_s = {time.time()-t0:0.1f}")

imgs = sorted(p for p in glob.glob(os.path.join(RAW, "*")) if _is_img(p))
print(f"REPORT step = AUTOLABEL | n_raw_images = {len(imgs)} | dir = {RAW}")
assert len(imgs) > 0, f"No PNG/JPG in {RAW} or {SEED}."

to_run = [p for p in imgs if not os.path.isfile(os.path.join(LABELS, os.path.splitext(os.path.basename(p))[0] + ".txt"))]
print(f"REPORT step = AUTOLABEL | already_labeled = {len(imgs)-len(to_run)} | will_label = {len(to_run)} | conf = {CONF} | iou = {IOU}")

if to_run:
    model_t = YOLO(TEACHER)
    t0 = time.time()
    n_det_total = 0
    n_imgs_with_det = 0
    for r in model_t.predict(source=to_run, conf=CONF, iou=IOU, save=False, verbose=False, stream=True):
        stem = os.path.splitext(os.path.basename(r.path))[0]
        out_txt = os.path.join(LABELS, stem + ".txt")
        lines = []
        boxes = r.boxes
        if boxes is not None and len(boxes) > 0:
            xywhn = boxes.xywhn.cpu().numpy()
            cls   = boxes.cls.cpu().numpy().astype(int)
            for i in range(len(boxes)):
                cx, cy, w, h = xywhn[i]
                lines.append(f"{int(cls[i])} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            n_imgs_with_det += 1
            n_det_total += len(boxes)
        with open(out_txt, "w") as fh:
            fh.write("\n".join(lines))
    print(f"REPORT step = AUTOLABEL | wrote = {len(to_run)} | total_dets = {n_det_total} | imgs_with_det = {n_imgs_with_det}/{len(to_run)} | elapsed_s = {time.time()-t0:0.1f}")
else:
    print("REPORT step = AUTOLABEL | status = SKIP_ALREADY_DONE")

ctr = Counter()
empty = 0
for lp in glob.glob(os.path.join(LABELS, "*.txt")):
    has_any = False
    with open(lp) as fh:
        for ln in fh:
            ln = ln.strip()
            if not ln: continue
            ctr[int(ln.split()[0])] += 1
            has_any = True
    if not has_any:
        empty += 1
for cid, name in enumerate(CLASSES):
    print(f"REPORT autolabel | class = {name:11s} | n = {ctr.get(cid, 0)}")
print(f"REPORT autolabel | total_instances = {sum(ctr.values())} | empty_label_files = {empty}")

## 6.2 — Visual sanity preview

Saves a 3×3 grid of randomly chosen raw screenshots with the auto-label boxes overlaid (color-coded by class). Open `desktop_autolabel_preview.png` from Drive — if you see e.g. an entire VS Code editor pane labeled as `text` or the Explorer tree treated as one big `menu`, that's expected (and it's fine for the prototype). If predictions are wildly off, lower `CONF` in 6.1 and re-run.


In [ ]:
import os, glob, random
import matplotlib.pyplot as plt
from ultralytics import YOLO

random.seed(0)
imgs = sorted(p for p in glob.glob(os.path.join(RAW, "*"))
              if p.lower().endswith((".png", ".jpg", ".jpeg")))
sample = random.sample(imgs, k=min(9, len(imgs)))

teacher = YOLO(TEACHER)
results = teacher.predict(source=sample, conf=CONF, iou=IOU,
                          save=False, verbose=False)

fig, axes = plt.subplots(3, 3, figsize=(22, 14))
for ax, r in zip(axes.flat, results):
    rgb = r.plot()[..., ::-1]
    ax.imshow(rgb); ax.axis("off")
    n = len(r.boxes) if r.boxes is not None else 0
    ax.set_title(f"{os.path.basename(r.path)}  ({n} dets)", fontsize=10)

plt.suptitle("Desktop auto-labels (teacher = best_source_v8s.pt, conf=0.20)", fontsize=12)
plt.tight_layout()
FIG_DIR = os.path.join(DRIVE, "reports", "figures"); os.makedirs(FIG_DIR, exist_ok=True)
out = os.path.join(FIG_DIR, "desktop_autolabel_preview.png")
plt.savefig(out, dpi=160, bbox_inches="tight"); plt.show()
print("REPORT step = PREVIEW | fig =", out)

teacher.predict(source=sample[:5], conf=CONF, iou=IOU, save=True,
                 project=os.path.join(DRIVE, "weights", "desktop_finetune"),
                 name="autolabel_preview_full", exist_ok=True, verbose=False)
full_dir = os.path.join(DRIVE, "weights", "desktop_finetune", "autolabel_preview_full")
print("REPORT step = PREVIEW_FULL | dir =", full_dir, "| open these PNGs at full resolution to eyeball labels")

## 6.3 — Build `/content/desktop_finetune/` (deterministic 70/15/15 split)

Copies the labelled images + labels into `/content/desktop_finetune/{images,labels}/{train,val,test}/` using a fixed `random.seed(42)` shuffle so the splits are reproducible. With 50 images this is roughly **35 / 8 / 7**.

We then bundle each split into `<DRIVE>/data/desktop_finetune_bundles/<split>.tar.gz` (images **and** labels — the dataset is only ~15 MB). On a fresh Colab tab, this lets the next notebook (or a re-run of this one) restore the dataset in <5 s without re-doing autolabel.

Skip-rules:
- Local `/content/desktop_finetune/` already has 50 items → skip.
- Drive bundles exist → fast-restore from bundles.
- Otherwise → build from scratch + persist bundles.


In [ ]:
import os, random, shutil, tarfile, time, glob

OUT       = "/content/desktop_finetune"
BUNDLES   = os.path.join(DRIVE, "data", "desktop_finetune_bundles")
DATA_YAML = os.path.join(OUT, "data.yaml")
SPLITS    = ["train", "val", "test"]
CLASSES   = ["button", "text", "text_input", "icon", "menu", "checkbox"]

def _count(p):
    try:
        n = 0
        with os.scandir(p) as it:
            for e in it:
                if e.is_file(follow_symlinks=False) or e.is_symlink():
                    n += 1
        return n
    except OSError:
        return 0

def _bundles_present():
    for sp in SPLITS:
        b = os.path.join(BUNDLES, f"{sp}.tar.gz")
        if not (os.path.isfile(b) and os.path.getsize(b) > 200):
            return False
    return True

have = {sp: _count(os.path.join(OUT, "images", sp)) for sp in SPLITS}
have_total = sum(have.values())
print("REPORT pre-check | local =", have, "| total =", have_total)

BUILT = False
if have_total >= 50:
    print("REPORT step = ASSEMBLE | status = ALREADY_BUILT")
    BUILT = True
elif _bundles_present():
    t0 = time.time()
    os.makedirs(OUT, exist_ok=True)
    for sp in SPLITS:
        with tarfile.open(os.path.join(BUNDLES, f"{sp}.tar.gz"), "r:gz") as tf:
            tf.extractall(OUT)
    print(f"REPORT step = ASSEMBLE | status = RESTORED_FROM_BUNDLE | elapsed_s = {time.time()-t0:0.1f}")
    BUILT = True

if not BUILT:
    imgs = sorted(p for p in glob.glob(os.path.join(RAW, "*")) if p.lower().endswith((".png", ".jpg", ".jpeg")))
    pairs = []
    for p in imgs:
        lp = os.path.join(LABELS, os.path.splitext(os.path.basename(p))[0] + ".txt")
        if os.path.isfile(lp):
            pairs.append((p, lp))
    print(f"REPORT step = ASSEMBLE | n_with_labels = {len(pairs)}/{len(imgs)}")
    assert len(pairs) >= 10, "Too few labelled images — run 6.1 first."

    rng = random.Random(42)
    order = sorted(pairs, key=lambda x: os.path.basename(x[0]))
    rng.shuffle(order)
    n = len(order)
    n_val  = max(1, round(n * 0.15))
    n_test = max(1, round(n * 0.15))
    n_train = n - n_val - n_test
    splits = {
        "train": order[:n_train],
        "val":   order[n_train:n_train + n_val],
        "test":  order[n_train + n_val:],
    }
    for sp, items in splits.items():
        img_dir = os.path.join(OUT, "images", sp); os.makedirs(img_dir, exist_ok=True)
        lbl_dir = os.path.join(OUT, "labels", sp); os.makedirs(lbl_dir, exist_ok=True)
        for ip, lp in items:
            shutil.copy2(ip, os.path.join(img_dir, os.path.basename(ip)))
            shutil.copy2(lp, os.path.join(lbl_dir, os.path.basename(lp)))
        print(f"REPORT step = ASSEMBLE | split = {sp:5s} | n = {len(items)}")

    os.makedirs(BUNDLES, exist_ok=True)
    for sp in SPLITS:
        b = os.path.join(BUNDLES, f"{sp}.tar.gz")
        with tarfile.open(b, "w:gz") as tf:
            for sub in ("images", "labels"):
                tf.add(os.path.join(OUT, sub, sp), arcname=os.path.join(sub, sp))
        print(f"REPORT step = PERSIST | split = {sp:5s} | bundle = {b} | bytes = {os.path.getsize(b)}")
    print(f"REPORT step = PERSIST | status = DONE | dir = {BUNDLES}")

import yaml as _yaml
with open(DATA_YAML, "w") as fh:
    _yaml.safe_dump({
        "path": OUT,
        "train": "images/train", "val": "images/val", "test": "images/test",
        "nc": len(CLASSES), "names": CLASSES,
    }, fh, sort_keys=False)
print("REPORT step = ASSEMBLE | wrote =", DATA_YAML)
print(open(DATA_YAML).read())

from collections import Counter
for sp in SPLITS:
    img_n = _count(os.path.join(OUT, "images", sp))
    lbl_n = _count(os.path.join(OUT, "labels", sp))
    inst  = Counter()
    for lp in glob.glob(os.path.join(OUT, "labels", sp, "*.txt")):
        with open(lp) as fh:
            for ln in fh:
                ln = ln.strip()
                if ln:
                    inst[int(ln.split()[0])] += 1
    print(f"REPORT final | split = {sp:5s} | images = {img_n:>3d} | labels = {lbl_n:>3d} | instances = {sum(inst.values())}")
    for cid, name in enumerate(CLASSES):
        print(f"REPORT final | split = {sp:5s} | class = {name:11s} | n = {inst.get(cid, 0)}")

## 6.4 — Head-only fine-tune (resume / skip)

Loads `best_source_v8s.pt` and trains only the detection head + neck (`freeze=10` locks the first 10 layers — backbone). With 35 train images this is fast (~5–10 min on T4).

Logic (mirrors `05.2`):
- `best.pt` exists in `weights/desktop_finetune/run1/` and not `FORCE_FRESH` → **skip**, just load.
- `last.pt` exists → **resume**.
- else → fresh from teacher.

Hyperparameters:
- `epochs = 20`, `imgsz = 640`, `batch = 8`, `optimizer = AdamW`, `lr0 = 1e-3`, `cos_lr = True`, `freeze = 10`.

Persistence: `project = <DRIVE>/weights/desktop_finetune` so checkpoints survive runtime restarts.


In [ ]:
import os, time, shutil
from ultralytics import YOLO

PROJECT = os.path.join(DRIVE, "weights", "desktop_finetune")
NAME    = "run1"
RUN_DIR = os.path.join(PROJECT, NAME)
LAST_PT = os.path.join(RUN_DIR, "weights", "last.pt")
BEST_PT = os.path.join(RUN_DIR, "weights", "best.pt")
TEACHER = os.path.join(DRIVE, "weights", "baseline_source", "best_source_v8s.pt")
os.makedirs(PROJECT, exist_ok=True)

FORCE_FRESH = False
EPOCHS = 20
IMGSZ  = 640
BATCH  = 8
LR0    = 1e-3
FREEZE = 10

t0 = time.time()
train_status = None

if os.path.isfile(BEST_PT) and not FORCE_FRESH:
    print("best.pt already exists -> skipping training and loading:", BEST_PT)
    model = YOLO(BEST_PT)
    train_status = "SKIP_ALREADY_TRAINED"
elif os.path.isfile(LAST_PT) and not FORCE_FRESH:
    print("last.pt found -> resuming:", LAST_PT)
    model = YOLO(LAST_PT)
    model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=2,
        optimizer="AdamW", lr0=LR0, cos_lr=True, patience=10, freeze=FREEZE,
        project=PROJECT, name=NAME, exist_ok=True,
        cache=True, seed=0, save=True, plots=True, resume=True,
    )
    train_status = "RESUMED"
else:
    assert os.path.isfile(TEACHER), f"Missing teacher: {TEACHER} — run 05 first."
    print(f"starting from teacher (best_source_v8s.pt) with freeze={FREEZE}")
    model = YOLO(TEACHER)
    model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=2,
        optimizer="AdamW", lr0=LR0, cos_lr=True, patience=10, freeze=FREEZE,
        project=PROJECT, name=NAME, exist_ok=True,
        cache=True, seed=0, save=True, plots=True,
    )
    train_status = "FRESH"

elapsed = time.time() - t0
print(f"REPORT step = TRAIN | status = {train_status} | elapsed_s = {elapsed:0.0f} | run_dir = {RUN_DIR}")
print(f"REPORT step = TRAIN | epochs = {EPOCHS} | imgsz = {IMGSZ} | batch = {BATCH} | lr0 = {LR0} | freeze = {FREEZE} | optimizer = AdamW")

## 6.5 — Save a stable name for `best.pt`

The demo notebook references `desktop_finetune/best_desktop_v8s.pt` without needing to know the run number.


In [ ]:
import os, shutil
STABLE = os.path.join(PROJECT, "best_desktop_v8s.pt")
if os.path.isfile(BEST_PT):
    shutil.copy2(BEST_PT, STABLE)
    print("REPORT step = SAVE_STABLE | src =", BEST_PT, "| dst =", STABLE, "| bytes =", os.path.getsize(STABLE))
else:
    print("REPORT step = SAVE_STABLE | status = MISSING_BEST")

## 6.6 — Test-split metrics + sanity predictions + per-class CSV

Runs `model.val(split="test")` on the 7 held-out screenshots, writes:
- `<DRIVE>/reports/tables/desktop_finetune_results.csv` — one-row summary for §4.
- `<DRIVE>/reports/tables/desktop_finetune_per_class.csv` — P/R/AP50/mAP50-95 per class for §4.5.
- Sanity prediction PNGs at `<DRIVE>/weights/desktop_finetune/predict_test/`.
- Copy of training curves to `<DRIVE>/reports/figures/desktop_finetune_curves.png`.

> Caveat: the test split labels are also auto-generated (since we have no human GT), so these metrics measure **agreement with the teacher**, not absolute correctness. They're fine for tracking improvement vs. the source baseline; for a dissertation-grade number, hand-correct the 7 test labels in Roboflow and re-run this cell.


In [ ]:
import os, csv, glob, time, shutil

CLASSES = ["button", "text", "text_input", "icon", "menu", "checkbox"]
TABLES  = os.path.join(DRIVE, "reports", "tables");  os.makedirs(TABLES, exist_ok=True)
FIGS    = os.path.join(DRIVE, "reports", "figures"); os.makedirs(FIGS, exist_ok=True)

val_m = model.val(data=DATA_YAML, split="test",
                   project=PROJECT, name="val_test", exist_ok=True,
                   plots=False, verbose=True)
box = getattr(val_m, "box", None)

row = {
    "model": "YOLOv8s",
    "dataset": "desktop_finetune (50 personal screenshots, auto-labelled)",
    "init_weights": "best_source_v8s.pt",
    "freeze": FREEZE,
    "n_train": _count(os.path.join(OUT, "images", "train")),
    "n_val":   _count(os.path.join(OUT, "images", "val")),
    "n_test":  _count(os.path.join(OUT, "images", "test")),
    "imgsz": IMGSZ, "batch": BATCH, "epochs": EPOCHS, "lr0": LR0,
    "map50":     float(getattr(box, "map50", float("nan"))) if box is not None else float("nan"),
    "map50_95":  float(getattr(box, "map",   float("nan"))) if box is not None else float("nan"),
    "precision": float(getattr(box, "mp",    float("nan"))) if box is not None else float("nan"),
    "recall":    float(getattr(box, "mr",    float("nan"))) if box is not None else float("nan"),
    "weights": STABLE if os.path.isfile(STABLE) else BEST_PT,
    "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
}
csv_path = os.path.join(TABLES, "desktop_finetune_results.csv")
with open(csv_path, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(row.keys()))
    w.writeheader(); w.writerow(row)
print("REPORT step = METRICS | csv =", csv_path)
for k, v in row.items():
    print(f"REPORT metric | {k:13s} = {v}")

try:
    if box is not None:
        idx = list(getattr(box, "ap_class_index", range(len(CLASSES))))
        p_arr  = list(getattr(box, "p",     []))
        r_arr  = list(getattr(box, "r",     []))
        ap50   = list(getattr(box, "ap50",  []))
        maps_  = list(getattr(box, "maps",  []))
        rows = []
        for cid, name in enumerate(CLASSES):
            try:
                pos = idx.index(cid) if cid in idx else None
            except ValueError:
                pos = None
            rows.append({
                "class": name,
                "P":        float(p_arr[pos])  if pos is not None and pos < len(p_arr)  else float("nan"),
                "R":        float(r_arr[pos])  if pos is not None and pos < len(r_arr)  else float("nan"),
                "AP50":     float(ap50[pos])   if pos is not None and pos < len(ap50)   else float("nan"),
                "mAP50_95": float(maps_[cid])  if cid < len(maps_)                       else float("nan"),
            })
        pcsv = os.path.join(TABLES, "desktop_finetune_per_class.csv")
        with open(pcsv, "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=["class","P","R","AP50","mAP50_95"])
            w.writeheader()
            for r in rows: w.writerow(r)
        print("REPORT step = PER_CLASS | csv =", pcsv)
        for r in rows:
            print(f"REPORT per_class | {r['class']:11s} | P = {r['P']:.4f} | R = {r['R']:.4f} | AP50 = {r['AP50']:.4f} | mAP50_95 = {r['mAP50_95']:.4f}")
except Exception as e:
    print("WARN: per-class CSV failed:", e)

test_imgs = sorted(glob.glob(os.path.join(OUT, "images", "test", "*")))
if test_imgs:
    model.predict(source=test_imgs, conf=0.25, save=True,
                   project=PROJECT, name="predict_test", exist_ok=True)
    pred_dir = os.path.join(PROJECT, "predict_test")
    print(f"REPORT step = PREDICT | n_images = {len(test_imgs)} | out_dir = {pred_dir}")

curve_src = os.path.join(RUN_DIR, "results.png")
if os.path.isfile(curve_src):
    dst = os.path.join(FIGS, "desktop_finetune_curves.png")
    shutil.copy2(curve_src, dst)
    print("REPORT step = FIG_COPY | src =", curve_src, "| dst =", dst)

**Next:** `notebooks/07_demo.ipynb` (§F) — interactive Streamlit/Gradio demo backed by `best_desktop_v8s.pt` for the dissertation video.

To improve numbers before §F:
1. Open `desktop_autolabel_preview.png` and the per-image labels in Roboflow.
2. Hand-correct the 7 test images (highest leverage) plus any obviously wrong train labels.
3. Re-upload corrected `.txt` files to `<DRIVE>/data/desktop/labels/`, delete `<DRIVE>/data/desktop_finetune_bundles/`, delete `/content/desktop_finetune/`, and re-run cells 6.3 → 6.6. The resulting test mAP becomes a real number you can quote in §4.
